Data exploration and feature engineering based on [Titanic - Decision Tree / RF / GB](https://www.kaggle.com/bcruise/titanic-decision-tree-rf-gb). This notebook builds on that by showing how to do parameter tuning using grid search to get better results with a gradient boost model.

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session
print('Setup complete.')

/kaggle/input/titanic/train.csv
/kaggle/input/titanic/test.csv
/kaggle/input/titanic/gender_submission.csv
Setup complete.


In [2]:
# Load the training dataset and look at the first few records.
train_df = pd.read_csv('/kaggle/input/titanic/train.csv')
print(train_df.shape)
train_df.head()

(891, 12)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
# Look at summary statistics for the training dataset.
train_df.describe()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


## Feature Engineering

In [4]:
train_df['Cabin'] = train_df['Cabin'].fillna('Unknown')
train_df.groupby(['Cabin']).apply(lambda x: x['Survived'].sum()/len(x))

Cabin
A10        0.000000
A14        0.000000
A16        1.000000
A19        0.000000
A20        1.000000
             ...   
F38        0.000000
F4         1.000000
G6         0.500000
T          0.000000
Unknown    0.299854
Length: 148, dtype: float64

In [5]:
train_df['n_Cabins'] = train_df.apply(lambda row: len(row['Cabin'].split()), axis=1)
train_df.groupby(['n_Cabins', 'Sex']).apply(lambda x: x['Survived'].sum()/len(x))

n_Cabins  Sex   
1         female    0.739274
          male      0.184397
2         female    0.714286
          male      0.444444
3         female    1.000000
          male      0.250000
4         female    1.000000
dtype: float64

In [6]:
train_df['CabinPrefix'] = train_df.apply(lambda row: row['Cabin'][0], axis=1)
train_df.groupby(['CabinPrefix', 'Sex']).apply(lambda x: x['Survived'].sum()/len(x))

CabinPrefix  Sex   
A            female    1.000000
             male      0.428571
B            female    1.000000
             male      0.400000
C            female    0.888889
             male      0.343750
D            female    1.000000
             male      0.466667
E            female    0.933333
             male      0.588235
F            female    1.000000
             male      0.375000
G            female    0.500000
T            male      0.000000
U            female    0.654378
             male      0.136170
dtype: float64

In [7]:
def split_ticket(ticket):
    # special case. a few Tickets are only LINE
    if ticket == 'LINE':
        return pd.Series(['LINE', 0])
    
    parts = ticket.split()  # split the ticket on whitespace
    if len(parts) == 1:
        return pd.Series(["NO_PREFIX", int(parts[0].strip(' .'))])
    elif len(parts) == 2:
        return pd.Series([parts[0].strip(' .'), int(parts[1].strip(' .'))])
    else:
        # special case. One Ticket has a prefix separated by a space.
        return pd.Series([parts[0].strip(' .') + parts[2].strip(' .'), int(parts[2].strip(' .'))])

train_df[['Ticket_Prefix', 'Ticket_NUM']] = train_df.apply(lambda row: split_ticket(row['Ticket']), axis=1)
train_df.groupby(['Ticket_Prefix']).apply(lambda x: x['Survived'].sum()/len(x))

Ticket_Prefix
A./5             0.000000
A.5              0.000000
A/4              0.000000
A/5              0.117647
A/S              0.000000
A4               0.000000
C                0.400000
C.A              0.481481
C.A./SOTON       0.000000
CA               0.071429
F.C              0.000000
F.C.C            0.800000
Fa               0.000000
LINE             0.250000
NO_PREFIX        0.384266
P/PP             0.500000
PC               0.650000
PP               0.666667
S.C./A.4         0.000000
S.C./PARIS       0.500000
S.O./P.P         0.000000
S.O.C            0.000000
S.O.P            0.000000
S.P              0.000000
S.W./PP          1.000000
SC               1.000000
SC/AH            0.500000
SC/AH541         1.000000
SC/PARIS         0.400000
SC/Paris         0.500000
SCO/W            0.000000
SO/C             1.000000
SOTON/O.Q        0.125000
SOTON/O2         0.000000
SOTON/OQ         0.142857
STON/O2          0.500000
STON/O3101269    1.000000
STON/O3101273    0.00000

In [8]:
train_df['Title'] = train_df.apply(lambda row: row['Name'].split()[1], axis=1)
train_df.groupby(['Title']).apply(lambda x: x['Survived'].sum()/len(x))

Title
Billiard,       0.000000
Capt.           0.000000
Carlo,          0.000000
Col.            0.500000
Cruyssen,       0.000000
Don.            0.000000
Dr.             0.428571
Gordon,         1.000000
Impe,           0.000000
Jonkheer.       0.000000
Major.          0.500000
Master.         0.575000
Melkebeke,      0.000000
Messemaeker,    1.000000
Miss.           0.703911
Mlle.           1.000000
Mme.            1.000000
Mr.             0.157371
Mrs.            0.801653
Ms.             1.000000
Mulder,         1.000000
Pelsmaeker,     0.000000
Planke,         0.000000
Rev.            0.000000
Shawah,         0.000000
Steen,          0.000000
Velde,          0.000000
Walle,          0.000000
der             0.000000
the             1.000000
y               0.750000
dtype: float64

In [9]:
from sklearn.preprocessing import LabelEncoder

# Select feaures to base the model on.
features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 
            'Embarked', 'n_Cabins', 'CabinPrefix', 'Ticket_Prefix', 'Ticket_NUM', 'Title']
X = train_df.loc[: ,features]
y = train_df['Survived']

# Scikit-learn doesn't like categorical features as strings, so we'll encode them as numbers.
X['Sex'] = X['Sex'].map( {'male':1, 'female':0} )

# Remember we had a couple of missing values in the Embarked column.
# We'll fill those with 'Unknown' before we try to encode this column
X['Embarked'] = X['Embarked'].fillna('Unknown')

# Age also had several missing values. Fill those with the average age.
X['Age'] = X['Age'].fillna(28.0)

emb_encoder = LabelEncoder()
emb_encoder.fit(X['Embarked'])
X['Embarked'] = emb_encoder.transform(X['Embarked'])

cab_encoder = LabelEncoder()
cab_encoder.fit(X['CabinPrefix'])
X['CabinPrefix'] = cab_encoder.transform(X['CabinPrefix'])

tic_encoder = LabelEncoder()
tic_encoder.fit(X['Ticket_Prefix'])
X['Ticket_Prefix'] = tic_encoder.transform(X['Ticket_Prefix'])

title_encoder = LabelEncoder()
title_encoder.fit(X['Title'])
X['Title'] = title_encoder.transform(X['Title'])
X

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,n_Cabins,CabinPrefix,Ticket_Prefix,Ticket_NUM,Title
0,3,1,22.0,1,0,7.2500,2,1,8,3,21171,17
1,1,0,38.0,1,0,71.2833,0,1,2,16,17599,18
2,3,0,26.0,0,0,7.9250,2,1,8,35,3101282,14
3,1,0,35.0,1,0,53.1000,2,1,2,14,113803,18
4,3,1,35.0,0,0,8.0500,2,1,8,14,373450,17
...,...,...,...,...,...,...,...,...,...,...,...,...
886,2,1,27.0,0,0,13.0000,2,1,8,14,211536,23
887,1,0,19.0,0,0,30.0000,2,1,1,14,112053,14
888,3,0,28.0,1,2,23.4500,2,1,8,49,6607,14
889,1,1,26.0,0,0,30.0000,0,1,2,14,111369,17


## Grid Search

Use [GridSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html) from scikit-learn to find the best parameters from an initial set. The un-tuned gradient boost model in the original notebook gave us 84.3% accuracy on the train/validation split using the same features. That dropped all the way to 77.5% when the same model was used on the full training set to create a submission. Hopefully parameter tuning will improve upon that score.

In [10]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV

model = GradientBoostingClassifier(random_state=0)

param_dict = {
    "n_estimators":[5,50,250,500],
    "max_depth":[1,3,5,7,9],
    "learning_rate":[0.01,0.1,1,10,100]
}

grid = GridSearchCV(model, param_grid=param_dict, cv=3, verbose=2, n_jobs=4)
grid.fit(X, y)

Fitting 3 folds for each of 100 candidates, totalling 300 fits


[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  33 tasks      | elapsed:    5.5s
[Parallel(n_jobs=4)]: Done 154 tasks      | elapsed:   33.6s
[Parallel(n_jobs=4)]: Done 300 out of 300 | elapsed:  1.1min finished


GridSearchCV(cv=3, estimator=GradientBoostingClassifier(random_state=0),
             n_jobs=4,
             param_grid={'learning_rate': [0.01, 0.1, 1, 10, 100],
                         'max_depth': [1, 3, 5, 7, 9],
                         'n_estimators': [5, 50, 250, 500]},
             verbose=2)

In [11]:
grid.best_params_

{'learning_rate': 0.01, 'max_depth': 5, 'n_estimators': 500}

In [12]:
grid.best_estimator_

GradientBoostingClassifier(learning_rate=0.01, max_depth=5, n_estimators=500,
                           random_state=0)

In [13]:
grid.best_score_

0.8327721661054994

## Final submission

In [14]:
train_df = pd.read_csv('/kaggle/input/titanic/train.csv')
test_df = pd.read_csv('/kaggle/input/titanic/test.csv')
print(test_df.shape)
test_df.head()

(418, 11)


,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


In [15]:
# Set aside the test PassengerId values for later.
passenger_ids = test_df['PassengerId']

# Add a fake value for Survived that we'll remove again later.
test_df['Survived'] = np.NaN
print(test_df.shape)
test_df.head()

(418, 12)


,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Survived
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q,NaN
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S,NaN
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q,NaN
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S,NaN
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S,NaN


In [16]:
combo_df = train_df.append(test_df)
print(combo_df.shape)

(1309, 12)


In [17]:
# Add the engineered features to the combined dataset
combo_df['Cabin'] = combo_df['Cabin'].fillna('Unknown')
combo_df['n_Cabins'] = combo_df.apply(lambda row: len(row['Cabin'].split()), axis=1)
combo_df['CabinPrefix'] = combo_df.apply(lambda row: row['Cabin'][0], axis=1)
combo_df[['Ticket_Prefix', 'Ticket_NUM']] = combo_df.apply(lambda row: split_ticket(row['Ticket']), axis=1)
combo_df['Title'] = combo_df.apply(lambda row: row['Name'].split()[1], axis=1)
combo_df.describe()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare,n_Cabins,Ticket_NUM
count,1309.000000,891.000000,1309.000000,1046.000000,1309.000000,1309.000000,1308.000000,1309.000000,1.309000e+03
mean,655.000000,0.383838,2.294882,29.881138,0.498854,0.385027,33.295479,1.046600,2.830713e+05
std,378.020061,0.486592,0.837836,14.413493,1.041658,0.865560,51.758668,0.287557,6.353943e+05
min,1.000000,0.000000,1.000000,0.170000,0.000000,0.000000,0.000000,1.000000,0.000000e+00
25%,328.000000,0.000000,2.000000,21.000000,0.000000,0.000000,7.895800,1.000000,1.356700e+04
50%,655.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200,1.000000,1.108130e+05
75%,982.000000,1.000000,3.000000,39.000000,1.000000,0.000000,31.275000,1.000000,3.470750e+05
max,1309.000000,1.000000,3.000000,80.000000,8.000000,9.000000,512.329200,4.000000,3.101317e+06


In [18]:
# Select feaures to base the model on.
features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 
            'Embarked', 'n_Cabins', 'CabinPrefix', 'Ticket_Prefix', 'Ticket_NUM', 'Title', 'Survived']
combo_df = combo_df[features]

# Scikit-learn doesn't like categorical features as strings, so we'll encode them as numbers.
combo_df['Sex'] = combo_df['Sex'].map( {'male':1, 'female':0} )

# Remember we had a couple of missing values in the Embarked column.
# We'll fill those with 'Unknown' before we try to encode this column
combo_df['Embarked'] = combo_df['Embarked'].fillna('Unknown')

# Age also had several missing values. Fill those with the median age.
combo_df['Age'] = combo_df['Age'].fillna(28.0)

# There was one missing Fare. Fill it with the median fare.
combo_df['Fare'] = combo_df['Fare'].fillna(14.45)

emb_encoder = LabelEncoder()
emb_encoder.fit(combo_df['Embarked'])
combo_df['Embarked'] = emb_encoder.transform(combo_df['Embarked'])

cab_encoder = LabelEncoder()
cab_encoder.fit(combo_df['CabinPrefix'])
combo_df['CabinPrefix'] = cab_encoder.transform(combo_df['CabinPrefix'])

tic_encoder = LabelEncoder()
tic_encoder.fit(combo_df['Ticket_Prefix'])
combo_df['Ticket_Prefix'] = tic_encoder.transform(combo_df['Ticket_Prefix'])

title_encoder = LabelEncoder()
title_encoder.fit(combo_df['Title'])
combo_df['Title'] = title_encoder.transform(combo_df['Title'])

combo_df.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,n_Cabins,CabinPrefix,Ticket_Prefix,Ticket_NUM,Title,Survived
0,3,1,22.0,1,0,7.2500,2,1,8,3,21171,19,0.0
1,1,0,38.0,1,0,71.2833,0,1,2,20,17599,20,1.0
2,3,0,26.0,0,0,7.9250,2,1,8,41,3101282,16,1.0
3,1,0,35.0,1,0,53.1000,2,1,2,18,113803,20,1.0
4,3,1,35.0,0,0,8.0500,2,1,8,18,373450,19,0.0


In [19]:
# Split the combined dataframe back into train and test data.
train_df = combo_df[combo_df['Survived'].notna()]
print(train_df.shape)
train_df.describe()

(891, 13)


,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,n_Cabins,CabinPrefix,Ticket_Prefix,Ticket_NUM,Title,Survived
count,891.000000,891.000000,891.000000,891.000000,891.000000,891.000000,891.000000,891.000000,891.000000,891.000000,8.910000e+02,891.000000,891.000000
mean,2.308642,0.647587,29.361582,0.523008,0.381594,32.204208,1.538721,1.038159,6.716049,19.189675,2.969891e+05,18.159371,0.383838
std,0.836071,0.477990,13.019697,1.102743,0.806057,49.693429,0.794231,0.252410,2.460739,8.107490,6.564383e+05,2.987872,0.486592
min,1.000000,0.000000,0.420000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000e+00,0.000000,0.000000
25%,2.000000,0.000000,22.000000,0.000000,0.000000,7.910400,1.000000,1.000000,8.000000,18.000000,1.431250e+04,16.000000,0.000000
50%,3.000000,1.000000,28.000000,0.000000,0.000000,14.454200,2.000000,1.000000,8.000000,18.000000,1.120580e+05,19.000000,0.000000
75%,3.000000,1.000000,35.000000,1.000000,0.000000,31.000000,2.000000,1.000000,8.000000,18.000000,3.470820e+05,19.000000,1.000000
max,3.000000,1.000000,80.000000,8.000000,6.000000,512.329200,3.000000,4.000000,8.000000,61.000000,3.101317e+06,33.000000,1.000000


In [20]:
test_df = combo_df[combo_df['Survived'].isna()]
test_df = test_df.drop('Survived', axis=1)
print(test_df.shape)
test_df.describe()

(418, 12)


,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,n_Cabins,CabinPrefix,Ticket_Prefix,Ticket_NUM,Title
count,418.000000,418.000000,418.000000,418.000000,418.000000,418.000000,418.000000,418.000000,418.000000,418.000000,4.180000e+02,418.000000
mean,2.265550,0.636364,29.805024,0.447368,0.392344,35.576525,1.401914,1.064593,6.758373,18.966507,2.534044e+05,18.236842
std,0.841838,0.481622,12.667969,0.896760,0.981429,55.850107,0.854496,0.350594,2.443901,7.732570,5.876873e+05,3.142328
min,1.000000,0.000000,0.170000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,2.000000e+00,0.000000
25%,1.000000,0.000000,23.000000,0.000000,0.000000,7.895800,1.000000,1.000000,8.000000,18.000000,1.350800e+04,16.750000
50%,3.000000,1.000000,28.000000,0.000000,0.000000,14.454200,2.000000,1.000000,8.000000,18.000000,3.560950e+04,19.000000
75%,3.000000,1.000000,35.750000,1.000000,0.000000,31.471875,2.000000,1.000000,8.000000,18.000000,3.455002e+05,19.000000
max,3.000000,1.000000,76.000000,8.000000,9.000000,512.329200,2.000000,4.000000,8.000000,59.000000,3.101315e+06,33.000000


In [21]:
# Split training features and target
train_y = train_df['Survived']
train_X = train_df.drop('Survived', axis=1)

# create and fit the model
model = GradientBoostingClassifier(learning_rate=0.01, max_depth=5, n_estimators=500,
                           random_state=0)
model.fit(train_X, train_y)

# get predictions based on test data
y_pred = model.predict(test_df)

final_gb_predictions_df = pd.DataFrame({'PassengerId': passenger_ids, 'Survived': y_pred}, dtype=int)
final_gb_predictions_df.to_csv('final_GB_predictions.csv', index=False)